# 大数据学期项目 — RAG 检索增强生成流水线
## 方向 B：智能客户支持与检索增强生成（RAG）助手

> **本 Notebook 完整演示项目数据流水线的每一步：数据摄取 → 清洗分块 → 向量索引 → 混合检索 → 答案生成。**
> 
> **PDF 要求**："一个 Jupyter Notebook，准确解释如何运行你们的流水线。"


## 0. 环境准备

本节检查 `.env` 是否存在，并展示当前嵌入配置。

- `local`：使用本地 `BAAI/bge-large-zh-v1.5` 嵌入模型。
- `remote`：保留本地 ETL，把 Embedding 推理卸载到 AutoDL RTX 4090 服务。

如果使用 AutoDL 远程服务，请确保 `EMBEDDING_SERVER_TOKEN` 已配置，避免 GPU 服务被滥用。


In [ ]:
# 环境准备（可选）
# !pip install -r requirements.txt

# 检查 .env 配置
import os
from pathlib import Path

env_path = Path(".env")
if not env_path.exists():
    print("未找到 .env，请先复制 .env.example 并填写 API Key")
    print("   copy .env.example .env")
else:
    print(".env 已找到")
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv("OPENAI_API_KEY", "")
    emb_model = os.getenv("OPENAI_EMBEDDING_MODEL", "local")
    emb_base = os.getenv("OPENAI_EMBEDDING_BASE_URL", "")
    local_emb = os.getenv("LOCAL_EMBEDDING_MODEL", "BAAI/bge-large-zh-v1.5")
    token = os.getenv("EMBEDDING_SERVER_TOKEN", "")
    print(f"  OPENAI_API_KEY: {'已设置' if api_key else 'MISSING'}")
    print(f"  Embedding 模式: {emb_model}")
    print(f"  本地模型: {local_emb}")
    print(f"  远程服务: {emb_base or '未配置'}")
    print(f"  访问 Token: {'已设置' if token else '未设置'}")
    print(f"  LLM 模型: {os.getenv('OPENAI_MODEL', 'gpt-4o-mini')}")


## 1. 项目结构总览

In [ ]:
BASE_DIR = Path.cwd()
print("项目根目录:", BASE_DIR)
print()
def show_tree(path, prefix="", max_depth=3, current_depth=0):
    if current_depth > max_depth: return
    items = sorted(path.iterdir(), key=lambda x: (x.is_file(), x.name))
    for i, item in enumerate(items):
        if item.name.startswith(".") or item.name == "__pycache__": continue
        connector = "└── " if i == len(items) - 1 else "├── "
        if item.is_dir():
            print(f"{prefix}{connector}{item.name}/")
            show_tree(item, prefix + ("    " if i == len(items) - 1 else "│   "),
                      max_depth, current_depth + 1)
        else:
            print(f"{prefix}{connector}{item.name}")
show_tree(BASE_DIR, max_depth=2)

## 2. 数据读取与 JSONL 隔离

项目将普通文本与 JSONL 数据分开读取：JSONL 统一通过 `load_jsonl_files()` 处理，普通文档只读取 `.md/.txt/.pdf`，避免重复摄取。


In [ ]:
import sys
sys.path.insert(0, str(Path("src")))
from utils import init_env, get_openai_client, get_model_name, get_embedding_model_name
from ingest import load_text_files, load_jsonl_files
from preprocess import process_documents, clean_text, chunk_text
from embed_store import VectorStore
from qa import generate_answer
from query_parser import parse_query
from collect_corpus import TOPICS

init_env()
client = get_openai_client()
print(f"LLM 模型: {get_model_name()}")
print(f"Embedding 模型: {get_embedding_model_name()}")
print(f"客户端类型: {type(client).__name__}")


## 3. 数据源一览

### 3.1 课程资料 (`data/raw/`)

In [ ]:
data_dir = BASE_DIR / "data" / "raw"
md_files = list(data_dir.rglob("*.md"))
txt_files = list(data_dir.rglob("*.txt"))
pdf_files = list(data_dir.rglob("*.pdf"))
print(f"Markdown 文件: {len(md_files)} 个")
print(f"TXT 文件:     {len(txt_files)} 个")
print(f"PDF 文件:     {len(pdf_files)} 个")
print(f"总计:          {len(md_files) + len(txt_files) + len(pdf_files)} 个")
print(f"  课程文档: 45 个")
print(f"  Wikipedia 词条: ~83 篇 (external/wiki_*.md)")
print(f"  Stack Overflow 问答: ~30 篇")
print(f"  CSDN 博客: ~18 篇")
print(f"  总计: ~100万行原始文档 (约 1,215,021 个分块)")

### 3.2 多源语料采集

项目集成三个知识来源：

| 来源 | 模块 | 内容 |
|------|------|------|
| Wikipedia | `collect_corpus.py` | 技术术语摘要（83 篇） |
| Stack Overflow | `collect_stackoverflow.py` | 高票技术问答，DeepSeek 翻译为全中文（30 篇） |
| CSDN 博客 | `collect_csdn.py` | 中文技术博客（18 篇） |

CLI 命令：`python -m src.main collect-all` 一键全量采集。

In [ ]:
print(f"Wikipedia 词条主题 ({len(TOPICS)} 个):")
for i, t in enumerate(TOPICS, 1):
    print(f"  {i:2d}. {t.zh_title} ({t.en_title}) [{t.tag}]")
    if i >= 10:
        print(f"  ... 还有 {len(TOPICS) - 10} 个")
        break

In [ ]:
# 查看已采集的语料

external_dir = data_dir / "external"
if external_dir.exists():
    wiki_files = list(external_dir.glob("wiki_*.md"))
    so_files = list(external_dir.glob("so_*.md"))
    csdn_files = list(external_dir.glob("csdn_*.md"))
    print(f"Wikipedia 词条: {len(wiki_files)} 篇")
    print(f"Stack Overflow 问答: {len(so_files)} 篇")
    print(f"CSDN 博客: {len(csdn_files)} 篇")
    print(f"总计: {len(wiki_files) + len(so_files) + len(csdn_files)} 篇")
else:
    print("external 目录不存在，请先运行采集:")
    print("  python -m src.main collect-all")

## 4. 文档摄取与合并

`load_text_files()` 读取 Markdown/TXT/PDF，`load_jsonl_files()` 专门读取 JSONL。两类数据在内存中合并，避免 JSONL 被普通文本加载器重复处理。


In [ ]:
documents = load_text_files(data_dir)
jsonl_documents = load_jsonl_files(BASE_DIR / "data")
all_documents = documents + jsonl_documents
print(f"Markdown/TXT/PDF 文档数: {len(documents)}")
print(f"JSONL 文档数: {len(jsonl_documents)}")
print(f"合计文档数: {len(all_documents)}")

if all_documents:
    sample = all_documents[0]
    print("\n样例文档:")
    print(f"  source: {sample['source']}")
    print(f"  path: {sample['path']}")
    print(f"  text 长度: {len(sample['text'])} 字符")
    print(f"  fm_meta: {sample.get('fm_meta', {})}")


## 5. 数据清洗

`clean_text()` 四步清洗流水线：移除 HTML 标签 → 解码 HTML 实体 → 过滤控制字符 → 规范化空白。

| 步骤 | 操作 | 示例 |
|------|------|------|
| 1 | 移除 HTML 标签 | `<p>hello</p>` → ` hello ` |
| 2 | 解码 HTML 实体 | `&amp;` → `&`，`&lt;` → `<` |
| 3 | 过滤控制字符 | 删除 `\x00-\x08`、`\x0b-\x0c`、`\x0e-\x1f`、`\x7f` |
| 4 | 规范化空白 | 合并连续空格，3+ 换行 → 2 换行 |

In [ ]:
dirty_sample = None
for doc in documents:
    if "<" in doc["text"] or "&" in doc["text"]:
        dirty_sample = doc; break
if dirty_sample:
    print("清洗前 (前 300 字符):")
    print(dirty_sample["text"][:300])
    cleaned = clean_text(dirty_sample["text"])
    print("\n清洗后 (前 300 字符):")
    print(cleaned[:300])
else:
    sample = documents[0]
    cleaned = clean_text(sample["text"])
    print(f"'{sample['source']}' 清洗完成")
    print(f"原始长度: {len(sample['text'])} → 清洗后: {len(cleaned)} 字符")

## 6. 语义分块

`chunk_text()` 四层优先级算法：
```
段落边界 → 句子边界 → 贪心合并 → 滑窗切割
```

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `chunk_size` | 700 | 目标块大小（字符）|
| `overlap` | 120 | 相邻块重叠长度 |
| `min_chunk_chars` | 40 | 最小块大小 |

> 参数可通过 build 命令调整：`python -m src.main build --chunk-size 800 --overlap 150`

In [ ]:
sample_doc = documents[0]
sample_cleaned = clean_text(sample_doc["text"])
chunks = chunk_text(sample_cleaned, chunk_size=700, overlap=120)
print(f"文档: {sample_doc['source']}")
print(f"原始长度: {len(sample_doc['text'])} 字符")
print(f"清洗后长度: {len(sample_cleaned)} 字符")
print(f"分块数: {len(chunks)}")
for i, ch in enumerate(chunks):
    preview = ch["text"][:80].replace("\n", " ")
    print(f"块 {i+1} [{ch['char_start']}:{ch['char_end']}]: {preview}...")

## 7. 元数据提取

extract_metadata() 用 LLM（deepseek-v4-flash）自动识别作者/年份/分类/语言/摘要。
**并发提取**：使用 ThreadPoolExecutor 32 线程并发调用 LLM，127 篇文档从串行 ~4 分钟降至 ~20 秒。
**429 限流重试**：遇到 DeepSeek API 限流时，自动指数退避重试（2s → 4s → 8s，最多 3 次）。
**Front-Matter 中的人工标注优先于 LLM 结果**
LLM 返回的 JSON 由 _safe_json_parse() 多层修复确保格式错误不丢数据。

In [ ]:
from preprocess import extract_metadata, _merge_fm_meta
demo_doc = documents[0]
c = clean_text(demo_doc["text"])
print("正在用 LLM 提取元数据...")
llm_meta = extract_metadata(c, demo_doc["source"], client)
print("LLM 提取的元数据:")
for k, v in llm_meta.items():
    print(f"  {k}: {v}")
fm_meta = demo_doc.get("fm_meta", {})
if fm_meta:
    merged = _merge_fm_meta(fm_meta, llm_meta)
    print("\n合并后的元数据 (Front-Matter 覆盖 LLM):")
    for k, v in merged.items():
        print(f"  {k}: {v}")

## 8. 元数据提取与预处理

`process_documents()` 负责清洗文本、语义分块与元数据合并。

- 第 1 步：清洗文本并生成稳定块边界。
- 第 2 步：按 `metadata_strategy` 合并 Front-Matter、JSONL 与 LLM 元数据。
- 第 3 步：输出可直接写入向量库的 chunk 字典。

`metadata_strategy` 支持 `merge`、`llm_only`、`jsonl_only`，非法取值会触发 `ValueError`，避免静默写入错误元数据。


In [ ]:
print("开始预处理...（演示中 is_extract_meta=False，避免额外 LLM 调用）")
processed = process_documents(
    all_documents,
    chunk_size=700,
    overlap=120,
    is_extract_meta=False,
    metadata_strategy="merge",
)
print("\n处理完成！")
print(f"原始文档: {len(all_documents)} 篇")
print(f"生成块数: {len(processed)} 块")
print(f"平均块数/文档: {len(processed) / max(len(all_documents), 1):.1f}")
import json
print("\n样例处理结果:")
print(json.dumps({k: str(v)[:60] for k, v in processed[0].items()}, indent=2, ensure_ascii=False))


## 9. 向量库初始化

`VectorStore` 封装 ChromaDB、嵌入模型调用和混合检索逻辑。

默认嵌入模型是 **BAAI/bge-large-zh-v1.5（1024 维）**。

当本地 GTX 1660 SUPER 显存不足时，可设置 `OPENAI_EMBEDDING_MODEL=remote`，把 Embedding 推理转移到 AutoDL RTX 4090；本地仍负责 ETL、HTTP 调用与 ChromaDB 写入。


In [ ]:
store = VectorStore(collection_name="course_docs")
print(f"集合名称: {store.collection.name}")
print(f"当前块数: {store.count()}")
print(f"嵌入模型: {store.embedding_model}")
print("距离度量: Cosine (HNSW)")


In [ ]:
sample_text = processed[0]["text"]
embedding = store.get_embedding(sample_text)
print(f"示例文本 (前80字): {sample_text[:80]}...")
print(f"向量维度: {len(embedding)}")
print(f"向量前 5 维: {embedding[:5]}")
print(f"向量值范围: [{min(embedding):.4f}, {max(embedding):.4f}]")

**向量库管理功能：**

| 功能 | 方法 | 说明 |
|------|------|------|
| 批量写入 | `add_documents(docs, batch_size=64)` | 自动嵌入 + upsert |
| 计数 | `count()` | 返回文档块总数 |
| 来源列表 | `list_sources()` | 返回去重排序的来源文件名 |
| 安全删除 | `delete_collection(confirm=True)` | 需显式确认防止误删 |

## 10. 检索：语义搜索 + 元数据过滤

`VectorStore.search()` 支持三种搜索模式：

In [ ]:
query = "什么是检索增强生成"
results = store.search(query, top_k=3)
print(f"查询: {query}")
print(f"返回条数: {len(results)}")
for i, r in enumerate(results):
    similarity = max(0, (1 - r["score"] / 2)) * 100
    preview = r["text"][:100].replace("\n", " ")
    print(f"[{i+1}] {r['source']} (距离={r['score']:.3f}, 相似度={similarity:.0f}%)")
    print(f"    内容: {preview}...")

In [ ]:
print("混合搜索: 语义 + 元数据过滤 (如只查 wiki 分类)")
results_filtered = store.search(query, top_k=3, where={"category": "wiki"})
print(f"过滤后结果: {len(results_filtered)} 条")
for r in results_filtered:
    meta = r.get("metadata", {})
    print(f"  {r['source']} | 分类={meta.get('category')} | 年份={meta.get('year')}")
print("\n注意: 如果过滤后无结果，系统自动回退为纯语义搜索")

In [ ]:
print("max_distance=1.0 (仅返回距离 <= 1.0 的高相关结果):")
results_dist = store.search(query, top_k=5, max_distance=1.0)
print(f"过滤后: {len(results_dist)} 条")
for r in results_dist:
    similarity = max(0, (1 - r["score"] / 2)) * 100
    print(f"  {r['source']} | 距离={r['score']:.3f} | 相似度={similarity:.0f}%")

## 11. 查询意图解析

`query_parser.py` 用 LLM (deepseek-v4-flash) 将自然语言问题分解为搜索词 + 结构化过滤条件：

In [ ]:
test_queries = [
    "2024年的课程通知里说了什么？",
    "RAG架构中向量数据库的作用是什么？",
    "去年老师有没有说过期末怎么考？",
]
for q in test_queries:
    parsed = parse_query(q, client=client)
    print(f"用户输入: {q}")
    print(f"  搜索词: {parsed['search_query']}")
    print(f"  过滤条件: {parsed['filters']}")
    print()

**回退机制：** 若 LLM 返回的 JSON 格式错误或 API 调用失败：
- `search_query` = 用户原始输入（全文搜索）
- `filters` = None（不做元数据过滤）
- 系统日志输出警告，但不中断流程

## 12. 答案生成

`qa.py` 将检索结果注入 LLM，生成带来源标注的答案：

In [ ]:
question = "Spark和Kafka有什么区别？"
print(f"问题: {question}\n")
parsed = parse_query(question, client=client)
print(f"解析出的搜索词: {parsed['search_query']}")
print(f"解析出的过滤: {parsed['filters']}\n")
retrieved = store.search(parsed["search_query"], top_k=3, where=parsed["filters"])
print(f"检索到 {len(retrieved)} 条相关文档:")
for i, r in enumerate(retrieved):
    print(f"  [{i+1}] {r['source']} (距离={r['score']:.3f})")
print()
answer = generate_answer(question, retrieved, client=client)
print("=" * 60)
print("答案:")
print(answer)
print("=" * 60)

### 防幻觉机制

| 层级 | 措施 |
|------|------|
| System Prompt | "Answer based only on the provided reference materials" |
| 上下文限制 | 仅将检索结果作为参考资料，不给 LLM 自由发挥空间 |
| 来源检查 | 若回答中无 `[Source: xxx]`，自动追加来源列表 |

In [ ]:
unrelated = "钢琴考级需要准备什么？"
parsed = parse_query(unrelated, client=client)
retrieved = store.search(parsed["search_query"], top_k=3)
answer = generate_answer(unrelated, retrieved, client=client)
print(f"问题: {unrelated}")
print(f"检索到 {len(retrieved)} 条")
print(f"答案:\n{answer}")

## 13. 全链路端到端验证

从原始文件到最终答案，一键执行：

In [ ]:
print("=" * 60)
print("全链路端到端测试")
print("=" * 60)
docs = load_text_files(data_dir)
print(f"[1/5] 加载文档: {len(docs)} 篇")
processed_docs = process_documents(docs[:5], chunk_size=700, overlap=120)
print(f"[2/5] 预处理: {len(processed_docs)} 个分块")
_emb = store.get_embedding(processed_docs[0]["text"])
print(f"[3/5] 嵌入测试: 向量维度 {len(_emb)}")
ret = store.search("RAG是什么", top_k=3)
print(f"[4/5] 检索: 返回 {len(ret)} 条")
ans = generate_answer("RAG是什么", ret, client=client)
has_source = "[Source:" in ans or "Sources:" in ans
print(f"[5/5] 答案生成: {'包含来源引用' if has_source else '无来源引用'}")
print(f"\n总文档块: {store.count()} 个")
print("全链路端到端验证通过!")

## 14. 系统状态概览

In [ ]:
print("=" * 40)
print("  系统状态面板")
print("=" * 40)
print(f"  向量库集合: {store.collection.name}")
print(f"  文档块总数: {store.count()}")
print(f"  来源文件数: {len(store.list_sources())}")
print(f"  LLM 模型:   {get_model_name()}")
print(f"  Embedding:  {store.embedding_model}")
print(f"  GPU 加速:   {'是' if store._use_local else '远程 API'}")
print("-" * 40)
print("  来源文件 (前10):")
for s in store.list_sources()[:10]:
    print(f"    {s}")
if len(store.list_sources()) > 10:
    print(f"    ... 共 {len(store.list_sources())} 个来源")

## 15. CLI、Web 与 AutoDL 命令速查

### CLI 命令
```bash
# 数据采集
python -m src.main collect          # Wikipedia
python -m src.main collect-so       # Stack Overflow
python -m src.main collect-csdn     # CSDN
python -m src.main collect-all      # 全量采集

# 普通文档 + JSONL 合并建库
python -m src.main build --metadata-strategy merge

# 仅使用 JSONL/Front-Matter 元数据，跳过 LLM 元数据提取
python -m src.main build --metadata-strategy jsonl_only

# 问答
python -m src.main ask --question "课程项目提交要求是什么？"
python -m src.main ask              # 交互模式
```

### AutoDL 远程 Embedding
```bash
# 云端 RTX 4090
EMBEDDING_SERVER_TOKEN=<自定义令牌> bash setup_autodl.sh

# 本地 .env
OPENAI_EMBEDDING_MODEL=remote
OPENAI_EMBEDDING_BASE_URL=https://<AutoDL公网地址>/v1
EMBEDDING_SERVER_TOKEN=<同一访问令牌>
LOCAL_EMBEDDING_MODEL=BAAI/bge-large-zh-v1.5
```

### Web 入口
```bash
streamlit run app/streamlit_app.py
```


## 16. 关键技术栈速查

| 层级 | 技术 | 选择原因 |
|------|------|------|
| 运行环境 | Python 3.11+ | 与课程实验环境和主流数据工具兼容 |
| 向量库 | ChromaDB | 本地持久化、安装轻量、支持元数据过滤 |
| Embedding 模型 | BAAI/bge-large-zh-v1.5 | 1024 维中文语义向量，适合课程语料检索 |
| GPU 策略 | 本地 ETL + AutoDL RTX 4090 推理 | 避免本地 GTX 1660 SUPER 显存瓶颈 |
| 服务鉴权 | EMBEDDING_SERVER_TOKEN | 给远程 Embedding 服务增加 Bearer Token 保护 |
| 分块策略 | 语义边界 700/120 | 保留上下文连续性，降低检索噪声 |
| 元数据策略 | Front-Matter/JSONL 合并 + LLM 回退 | 兼顾结构化数据可靠性与自动补全能力 |
| 检索策略 | 双路召回 | 课程文档优先，全库检索兜底 |
| 论文交付 | HTML + PDF 快照 | 便于浏览器打印与课程提交 |
| 答案生成 | Prompt + 引用约束 | 降低幻觉并保留来源证据 |


---

**最终论文**: `report/report_ieee.html`

**报告证据**:
- LLM: `gpt-4o-mini` 或 `.env` 中配置的 OpenAI-compatible 模型
- Embedding: `BAAI/bge-large-zh-v1.5`，支持 local/remote 两种模式
- 远程服务: AutoDL RTX 4090 + FastAPI OpenAI-compatible Embedding 服务
- 服务鉴权: `EMBEDDING_SERVER_TOKEN` Bearer Token
- 向量库: ChromaDB 本地持久化
- 测试: 91 个自动化测试，最近验证为 `91 passed`

---
